# Model Training and Final Evaluation

This notebook tunes the two strongest candidates from feature engineering, compares the complete improvement chain, evaluates the final winner once on the held-out test set, and saves reproducible model artifacts.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import shutil

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data" / "interim" / "X_train_engineered_scaled.csv").exists()
)
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"
PREPROCESSING_PATH = PROJECT_ROOT / "models" / "preprocessing"
TRAINED_PATH = PROJECT_ROOT / "models" / "trained"
METADATA_PATH = PROJECT_ROOT / "models" / "metadata"
EVALUATION_PATH = PROJECT_ROOT / "reports" / "evaluation"
FIGURES_PATH = PROJECT_ROOT / "reports" / "figures"
for output_path in (TRAINED_PATH, METADATA_PATH, EVALUATION_PATH, FIGURES_PATH):
    output_path.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(INTERIM_PATH / "X_train_engineered_scaled.csv")
X_test = pd.read_csv(INTERIM_PATH / "X_test_engineered_scaled.csv")
y_train = pd.read_csv(INTERIM_PATH / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(INTERIM_PATH / "y_test.csv").squeeze("columns")

feature_columns = list(X_train.columns)
expected_ratio_features = [
    "water_cement_ratio", "water_binder_ratio", "cement_binder_ratio",
    "slag_binder_ratio", "fly_ash_binder_ratio", "superplasticizer_binder_ratio",
    "aggregate_binder_ratio", "water_total_mix_ratio",
]
assert len(feature_columns) == 16
assert feature_columns == list(X_test.columns)
assert feature_columns[8:] == expected_ratio_features
assert X_train.shape == (804, 16)
assert X_test.shape == (201, 16)
assert not X_train.isna().any().any() and not X_test.isna().any().any()
assert not np.isinf(X_train.to_numpy()).any() and not np.isinf(X_test.to_numpy()).any()
assert "compressive_strength" not in feature_columns

saved_scaler = joblib.load(PREPROCESSING_PATH / "feature_scaler.joblib")
assert list(saved_scaler.feature_names_in_) == feature_columns

cv = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"Validated {X_train.shape[0]} training rows and {X_test.shape[0]} test rows.")
print(f"Using {len(feature_columns)} selected ratio features.")

In [ ]:
X_train_original = pd.read_csv(INTERIM_PATH / "X_train.csv")
X_test_original = pd.read_csv(INTERIM_PATH / "X_test.csv")

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

def cv_metrics(estimator, features):
    scores = cross_validate(estimator, features, y_train, cv=cv, scoring=scoring, n_jobs=1)
    return {
        "mae": float(-scores["test_mae"].mean()),
        "rmse": float(-scores["test_rmse"].mean()),
        "r2": float(scores["test_r2"].mean()),
        "mae_std": float(scores["test_mae"].std()),
        "rmse_std": float(scores["test_rmse"].std()),
        "r2_std": float(scores["test_r2"].std()),
    }

def pipeline(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])

baseline_original = cv_metrics(pipeline(GradientBoostingRegressor(random_state=42)), X_train_original)
baseline_engineered = cv_metrics(pipeline(GradientBoostingRegressor(random_state=42)), X_train)

searches = {
    "gradient_boosting": RandomizedSearchCV(
        pipeline(GradientBoostingRegressor(random_state=42)),
        {
            "model__n_estimators": [100, 200, 400],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [2, 3, 4],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__subsample": [0.8, 1.0],
        },
        n_iter=30,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=42,
        n_jobs=1,
        refit=True,
    ),
    "random_forest": RandomizedSearchCV(
        pipeline(RandomForestRegressor(random_state=42, n_jobs=1)),
        {
            "model__n_estimators": [200, 400, 700],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": [1.0, "sqrt"],
        },
        n_iter=30,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=42,
        n_jobs=1,
        refit=True,
    ),
}

tuned_models = {}
tuned_metrics = {}
for name, search in searches.items():
    search.fit(X_train, y_train)
    tuned_models[name] = search
    tuned_metrics[name] = cv_metrics(search.best_estimator_, X_train)
    print(f"{name}: {search.best_params_}")
    print(tuned_metrics[name])

winner_name = min(tuned_metrics, key=lambda name: tuned_metrics[name]["rmse"])
runner_up_name = max(
    (name for name in tuned_metrics if name != winner_name),
    key=lambda name: tuned_metrics[name]["rmse"],
)
winning_search = tuned_models[winner_name]
winning_pipeline = winning_search.best_estimator_
winning_model = winning_pipeline.named_steps["model"]
winning_cv = tuned_metrics[winner_name]
rmse_margin = tuned_metrics[runner_up_name]["rmse"] - winning_cv["rmse"]
combined_rmse_std = winning_cv["rmse_std"] + tuned_metrics[runner_up_name]["rmse_std"]

improvement_chain = pd.DataFrame([
    {"stage": "baseline_original_8_features_default_gbm", **baseline_original},
    {"stage": "engineered_16_ratios_default_gbm", **baseline_engineered},
    {"stage": "engineered_16_ratios_tuned_gbm", **tuned_metrics["gradient_boosting"]},
    {"stage": "engineered_16_ratios_tuned_random_forest", **tuned_metrics["random_forest"]},
])
print(improvement_chain.to_string(index=False))
print(f"Winner: {winner_name}; RMSE margin: {rmse_margin:.4f}; combined std: {combined_rmse_std:.4f}")

winning_pipeline.fit(X_train, y_train)
y_pred = winning_pipeline.predict(X_test)
assert y_pred.shape == y_test.shape
assert np.isfinite(y_pred).all()
residuals = y_test.to_numpy() - y_pred
final_test = {
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "r2": float(r2_score(y_test, y_pred)),
    "explained_variance": float(explained_variance_score(y_test, y_pred)),
    "mean_residual": float(residuals.mean()),
    "max_absolute_error": float(np.abs(residuals).max()),
}
print(f"Final test metrics: {final_test}")
improvement_chain = pd.concat([improvement_chain, pd.DataFrame([{"stage": "final_test_set_result", **final_test}])], ignore_index=True)
improvement_chain.to_csv(EVALUATION_PATH / "improvement_chain.csv", index=False)
pd.DataFrame({"actual": y_test, "predicted": y_pred, "residual": residuals}).to_csv(EVALUATION_PATH / "test_predictions.csv", index=False)

plots = [
    ("actual_vs_predicted.png", y_test, y_pred, "Actual", "Predicted"),
    ("residual_plot.png", y_pred, residuals, "Predicted", "Residual"),
]
for filename, x_values, y_values, xlabel, ylabel in plots:
    plt.figure(figsize=(7, 6))
    plt.scatter(x_values, y_values, alpha=0.7)
    if filename == "actual_vs_predicted.png":
        limits = [min(x_values.min(), y_values.min()), max(x_values.max(), y_values.max())]
        plt.plot(limits, limits, "--", color="black")
    else:
        plt.axhline(0, linestyle="--", color="black")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / filename, dpi=300)
    plt.close()

plt.figure(figsize=(7, 6))
plt.hist(residuals, bins=20, edgecolor="black")
plt.xlabel("Residual (MPa)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "residual_distribution.png", dpi=300)
plt.close()

ages = X_test_original["age"].to_numpy()
error_by_age = pd.DataFrame({"age": ages, "absolute_error": np.abs(residuals)}).groupby("age", as_index=False)["absolute_error"].mean()
strength_bins = pd.qcut(y_test, q=4, duplicates="drop").astype(str)
error_by_strength = pd.DataFrame({"strength_range": strength_bins, "absolute_error": np.abs(residuals)}).groupby("strength_range", as_index=False)["absolute_error"].mean()
error_by_age.to_csv(EVALUATION_PATH / "error_by_age.csv", index=False)
error_by_strength.to_csv(EVALUATION_PATH / "error_by_strength_range.csv", index=False)

built_in = pd.DataFrame({"feature": feature_columns, "importance": winning_model.feature_importances_}).sort_values("importance", ascending=False)
built_in.to_csv(EVALUATION_PATH / "feature_importance.csv", index=False)
permutation = permutation_importance(winning_pipeline, X_test, y_test, scoring="neg_root_mean_squared_error", n_repeats=20, random_state=42, n_jobs=1)
pd.DataFrame({"feature": feature_columns, "importance_mean": permutation.importances_mean, "importance_std": permutation.importances_std}).sort_values("importance_mean", ascending=False).to_csv(EVALUATION_PATH / "permutation_importance.csv", index=False)

model_version = "v1"
model_path = TRAINED_PATH / f"{winner_name}_model_{model_version}.joblib"
latest_path = TRAINED_PATH / "latest_model.joblib"
joblib.dump(winning_pipeline, model_path)
shutil.copyfile(model_path, latest_path)
reloaded = joblib.load(model_path)
assert np.allclose(y_pred, reloaded.predict(X_test))

metadata = {
    "model_name": winner_name,
    "model_version": model_version,
    "feature_set": "ratios",
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "random_state": 42,
    "best_parameters": winning_search.best_params_,
    "candidate_cv_metrics": tuned_metrics,
    "final_test_metrics": final_test,
    "untuned_baselines": {"original_8_features_default_gbm": baseline_original, "engineered_16_ratios_default_gbm": baseline_engineered},
    "rmse_margin_over_runner_up": rmse_margin,
    "combined_rmse_standard_deviation": combined_rmse_std,
    "scaling_note": "Tree models are scale-invariant; the saved scaler is retained for pipeline consistency.",
    "library_versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
    "scaler_path": str((PREPROCESSING_PATH / "feature_scaler.joblib").relative_to(PROJECT_ROOT)),
    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
    "latest_model_path": str(latest_path.relative_to(PROJECT_ROOT)),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_evaluation": "Held-out test metrics were calculated once before reload sanity checks.",
}
with open(METADATA_PATH / f"{winner_name}_metadata.json", "w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)
with open(EVALUATION_PATH / "model_metrics.json", "w", encoding="utf-8") as metrics_file:
    json.dump({"winner": winner_name, "cv": winning_cv, "test": final_test}, metrics_file, indent=2)
print(f"Saved model: {model_path}")
print("Model reload sanity check passed.")